# Cuaderno 9 — ¿Cuál es el mejor gráfico?

**Descripción y Visualización de Datos · UAI 2026 · Clase 9**

Hoy la regla es una sola: **primero la tarea, después el geom**. Antes de escribir
`ggplot()`, decide qué pregunta responde tu gráfico:

| Tarea | La pregunta | `aes()` | El geom |
|---|---|---|---|
| Ranking | ¿Cuál es más? | x = número, y = categoría | `geom_col()` + `reorder()` |
| Tendencia | ¿Cómo cambió? | x = tiempo, y = número | `geom_line()` + `geom_point()` |
| Distribución | ¿Cómo se reparte? | x = número, o x = grupo e y = número | `geom_histogram()`, `geom_boxplot()` |
| Composición | ¿De qué está hecho el total? | x = grupo, fill = parte | `geom_bar(position = "fill")` |
| Relación | ¿Van juntas? | x = número, y = número | `geom_point()`, `geom_jitter()`, `geom_smooth()` |
| Lugar | ¿Dónde pasa más? | x = número, y = lugar | `geom_point()` + `reorder()` |
| Cambio | ¿Quién se movió más? | x = antes, xend = después | `geom_segment()` + `geom_point()` |

Las Partes 1 a 7 son los siete ejemplos de las láminas: córrelos y prueba el cambio que
se pide en cada una. **La Parte 8 es el trabajo de hoy**, con la base de tu grupo.

## Parte 0 — Punto de partida

Tres bases: la CEP completa, la tabla de problemas por año y la encuesta del curso.

In [ ]:
library(dplyr)
library(ggplot2)
options(repr.plot.width = 9, repr.plot.height = 5)

cep       <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/cep_consolidada_1994_2026.csv")
problemas <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/cep_problemas_por_anio.csv")
curso     <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/encuesta_curso.csv")

curso <- curso %>%
  mutate(redes_num   = as.numeric(horas_redes),
         sueno_num   = as.numeric(horas_sueno),
         minutos_num = as.numeric(minutos_viaje))

## Parte 1 — Ranking: ¿cuál es más?

Las barras van **acostadas**, para que los nombres largos se lean, y `reorder()` las ordena de mayor a menor. Sin `reorder()`, salen en orden alfabético.

In [ ]:
top <- problemas %>%
  filter(anio == 2026) %>%
  arrange(desc(porcentaje)) %>%
  head(8)

ggplot(top, aes(x = porcentaje,
                y = reorder(problema, porcentaje))) +
  geom_col(fill = "#C4878C") +
  labs(
    title = "La delincuencia más que dobla a la salud",
    x = "% que lo nombra primer problema",
    y = NULL,
    caption = "Fuente: CEP 2026, n = 1.466") +
  theme_minimal()

**Prueba:** cambia `2026` por `1994`. ¿Sigue arriba la delincuencia?

In [ ]:
# Tu código acá

## Parte 2 — Tendencia: ¿cómo cambió?

El tiempo siempre en el eje x. La línea muestra la dirección; los puntos, cada medición. Fíjate entre 2019 y 2021: la línea **une dos años que no son seguidos**, porque en 2020 no hubo encuesta. Los puntos lo delatan.

In [ ]:
delinc <- problemas %>%
  filter(problema == "Delincuencia, asaltos y robos")

ggplot(delinc, aes(x = anio, y = porcentaje)) +
  geom_line(color = "#C4878C") +
  geom_point(color = "#C4878C") +
  labs(
    title = "Los tres últimos años, los más altos",
    x = NULL,
    y = "% que la nombra primer problema",
    caption = "Fuente: CEP 1994-2026, sin 2020") +
  theme_minimal()

**Prueba:** cambia la delincuencia por `"Salud"`. ¿Qué título le pondrías?

In [ ]:
# Tu código acá

## Parte 3 — Distribución: ¿cómo se reparte?

Un promedio esconde cómo se reparten los datos. La caja va del 25% al 75% de las personas y la raya del medio es la mediana.

In [ ]:
apr <- cep %>%
  filter(anio == 2026,
         aprueba_presidente %in%
           c("Aprueba", "Desaprueba"))

ggplot(apr, aes(x = aprueba_presidente, y = edad)) +
  geom_boxplot(fill = "#EBD6D9") +
  labs(
    title = "Los que desaprueban son más jóvenes",
    x = NULL,
    y = "Edad (años)",
    caption = "Fuente: CEP 2026, n = 1.260") +
  theme_minimal()

**Prueba:** la distribución de la edad de toda la muestra, con el histograma de la clase 8: `ggplot(apr, aes(x = edad)) + geom_histogram(binwidth = 5)`.

In [ ]:
# Tu código acá

## Parte 4 — Composición: ¿de qué está hecho el total?

`position = "fill"` estira cada barra al 100%: compara **proporciones**, no cantidades. El GSE E queda fuera porque tiene 9 casos (la trampa de la clase 6).

In [ ]:
apr3 <- cep %>%
  filter(anio == 2026, gse != "E",  # E: sólo 9 casos
         aprueba_presidente %in% c(
           "Aprueba", "Desaprueba",
           "No aprueba ni desaprueba"))

ggplot(apr3, aes(x = gse,
                 fill = aprueba_presidente)) +
  geom_bar(position = "fill") +
  scale_y_continuous(labels = scales::percent) +
  scale_fill_manual(
    values = c("#4F7396", "#B5646B", "#B5B0AA")) +
  labs(
    title = "Un tercio aprueba, en todos los GSE",
    x = "Grupo socioeconómico", y = NULL, fill = NULL,
    caption = "Fuente: CEP 2026, sin el GSE E") +
  theme_minimal() +
  theme(legend.position = "bottom")

**Prueba:** cambia `"fill"` por `"stack"`. ¿Qué gráfico responde mejor si la aprobación depende del GSE?

In [ ]:
# Tu código acá

## Parte 5 — Relación: ¿van juntas?

`geom_jitter()` mueve cada punto un poquito, para que las respuestas repetidas no queden una encima de otra (el problema de la clase 8). `geom_smooth(method = "lm")` dibuja la recta que mejor resume la nube. **Relación no es causa.**

In [ ]:
ggplot(curso, aes(x = redes_num, y = sueno_num)) +
  geom_jitter(width = 0.15, height = 0.15,
              color = "#C4878C", size = 2.5) +
  geom_smooth(method = "lm", se = FALSE,
              color = "#2C2C2C") +
  labs(
    title = "Más redes, algo menos de sueño",
    x = "Horas en redes al día",
    y = "Horas de sueño",
    caption = "Fuente: encuesta del curso, n = 32") +
  theme_minimal()

**Prueba:** cambia `redes_num` por `minutos_num`. ¿Duermen menos los que viajan más?

In [ ]:
# Tu código acá

## Parte 6 — Lugar: ¿dónde pasa más?

Un mapa necesita paquetes que todavía no vemos. Un ranking ordenado de lugares responde casi siempre lo mismo, y además muestra el **n** de cada uno.

In [ ]:
reg <- cep %>%
  filter(anio >= 2024) %>%
  mutate(delinc = ifelse(problema_1 ==
           "Delincuencia, asaltos y robos", 1, 0)) %>%
  group_by(region) %>%
  summarise(pct = mean(delinc) * 100, n = n()) %>%
  mutate(region = paste0(region, " (", n, ")"))

ggplot(reg, aes(x = pct, y = reorder(region, pct))) +
  geom_point(size = 3, color = "#C4878C") +
  labs(
    title = "Arica encabeza, pero con sólo 95 casos",
    x = "% que nombra primero la delincuencia",
    y = NULL,
    caption = "CEP 2024-2026; n entre paréntesis") +
  theme_minimal()

**Prueba:** cambia la delincuencia por `"Salud"`. ¿Cambia el orden de las regiones?

In [ ]:
# Tu código acá

## Parte 7 — Cambio: ¿quién se movió más?

Cuando la tabla tiene **una columna para antes y otra para después** (como `X2019` y `X2022`, o `Ano.2006` y `Ano.2024`), `geom_segment()` une los dos momentos y los puntos marcan cada extremo. Se ordena por el valor final.

In [ ]:
cambio <- data.frame(
  problema = c("Delincuencia", "Pobreza", "Salud",
               "Empleo", "Inflación", "Educación",
               "Sueldos"),
  en_1994 = c(19.0, 18.8, 13.4, 10.9, 9.6, 7.8, 6.1),
  en_2026 = c(31.5, 3.3, 13.2, 4.7, 1.4, 7.9, 2.6))

ggplot(cambio, aes(y = reorder(problema, en_2026))) +
  geom_segment(aes(x = en_1994, xend = en_2026,
                   yend = reorder(problema, en_2026)),
               color = "#B5B0AA") +
  geom_point(aes(x = en_1994), size = 3,
             color = "#B5B0AA") +
  geom_point(aes(x = en_2026), size = 3,
             color = "#C4878C") +
  labs(
    title = "Pobreza abajo, delincuencia arriba",
    x = "% primer problema: gris 1994, rosa 2026",
    y = NULL,
    caption = "Fuente: CEP 1994 y 2026") +
  theme_minimal()

**Prueba:** ordena por `en_1994` en vez de `en_2026`. ¿Cuál orden cuenta mejor la historia?

In [ ]:
# Tu código acá

## Parte 8 — El trabajo de hoy: tus gráficos

Con la base de tu grupo, la del Sprint 2. Para partir, cada grupo tiene una pista según
lo que entregó. **Es una pista, no una receta**: si encuentran un gráfico mejor y lo
defienden bien, el panel lo premia.

| Grupo | Tarea | Para empezar | Ojo con |
|---|---|---|---|
| G01 | Tendencia | `geom_line()` + `geom_point()`, con `color = Country.Name` | Los años no son seguidos (2017 → 2020): `Year` tiene que ser número. Gini y Palma tienen escalas distintas: un gráfico para cada uno, nunca dos ejes. |
| G02 | Composición | Una tabla con la parte de los cinco deportes con más fondos y el resto, por país; después `geom_col(position = "fill")` | Pesos y euros no se comparan: sólo porcentajes. En Irlanda cada deporte es una sola fila. |
| G03 | Composición | `geom_bar(position = "fill")` con `x = Anio` y `fill = Origen` | Las ejecuciones totales cambian mucho de un año a otro (2,4 millones en 2017, 158 mil en 2020): comparen proporciones, no totales. |
| G04 | Relación | Un año, un punto por país: `aes(x = Female, y = Male)` y `geom_abline(slope = 1)`, la línea donde hombres y mujeres serían iguales | Oceanía tiene 3 países. Elijan un año con muchos países reportados. |
| G05 | Ranking de cambios | `diferencia_pct` con `geom_col()` + `reorder()` | Saquen las filas «Total área…» antes de ordenar: no son ministerios. |
| G06 | Distribución | `geom_boxplot()` con `aes(x = factor(anio), y = tasa)` | `tasa_incidencia` llegó como texto: `as.numeric()`. Las comunas con muy pocos alumnos dan tasas extremas. |
| G07 | Cambio | `geom_segment(aes(x = Ano.2006, xend = Ano.2024, y = Pais, yend = Pais))` y dos `geom_point()` | Todo está entre 94% y 99,5%: unas barras que parten en cero se ven todas iguales. |
| G08 | Ranking | Las diez carreras con más matrícula: `geom_col()` + `reorder()` | Cada año tiene exactamente 1.048.575 filas, el máximo de Excel: revisen si la base quedó cortada antes de graficar. |
| G09 | Lugar | Regiones ordenadas: `geom_point()` + `reorder()` | Comparen tasas por habitante, no totales: en números absolutos la Metropolitana gana siempre. |

Carga tu base como en tu Colab:

In [ ]:
mi_base <- read.csv("PEGA ACÁ LA URL DE TU BASE")

glimpse(mi_base)

### Gráfico 1

**Tarea** (ranking, tendencia, distribución, composición, relación o lugar):

**Pregunta que responde:**

In [ ]:
# Tu código acá

### Gráfico 2

**Tarea** (ranking, tendencia, distribución, composición, relación o lugar):

**Pregunta que responde:**

In [ ]:
# Tu código acá

### Gráfico 3

**Tarea** (ranking, tendencia, distribución, composición, relación o lugar):

**Pregunta que responde:**

In [ ]:
# Tu código acá

### El que van a defender ante el panel

Elijan **uno** de los tres y hagan **otro gráfico con los mismos datos**: el que
descartaron. En el segundo bloque se muestran los dos, lado a lado.

**El elegido es el gráfico número:**

**El descartado es** (qué geom):

**Por qué el elegido responde mejor la pregunta** (una o dos líneas):

In [ ]:
# El descartado

## Para el panel

Roberto y los ayudantes van a mirar cuatro cosas:

| | Pregunta |
|---|---|
| **La tarea** | ¿Qué pregunta responde y es ese el tipo de gráfico que le corresponde? |
| **La lectura** | ¿Se entiende sin que nadie lo explique? |
| **El acabado** | Título con un hallazgo, ejes con unidad, fuente con n. |
| **La defensa** | ¿Por qué este y no el descartado? |

Tres minutos por grupo y dos de preguntas. Lo que construyan hoy es el primer borrador
del Sprint 3.